# Lab 3B: Caching and batches
**Claude Certified Developer (Foundations) · 30-Hour Program for UST · Day 3 · about 20 minutes**

**Objectives**
- Cache a long, stable prefix and prove it with usage fields
- See what breaks a cache and fix it
- Run a Message Batch and match results by custom_id
- Compare costs across standard, cached, and batched calls

**Before you start:** in Colab, click the key icon on the left, add a secret named `ANTHROPIC_API_KEY`, and turn on notebook access. All data in this lab is synthetic.

In [ ]:
# Setup: run once per session
!pip -q install -U anthropic 
import os, json, time, random, importlib, re
try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    assert os.environ.get("ANTHROPIC_API_KEY"), "Set ANTHROPIC_API_KEY in your environment"
import anthropic
print("anthropic SDK", anthropic.__version__, "| key loaded:", bool(os.environ.get("ANTHROPIC_API_KEY")))

In [ ]:
%%writefile config.py
"""One place for Claude settings (same pattern as Day 1)."""
import os
# Confirm current IDs on the models overview page: https://platform.claude.com/docs/en/about-claude/models/overview
MODEL = os.environ.get("CLAUDE_MODEL", "claude-sonnet-5")
FAST_MODEL = os.environ.get("CLAUDE_FAST_MODEL", "claude-haiku-4-5-20251001")
MAX_TOKENS = 1024
MAX_RETRIES = 2
TIMEOUT_S = 120

In [ ]:
import config
importlib.reload(config)
client = anthropic.Anthropic(max_retries=config.MAX_RETRIES, timeout=config.TIMEOUT_S)

def log_response(resp, label=""):
    kinds = [b.type for b in resp.content]
    print(f"{label} stop_reason={resp.stop_reason} blocks={kinds} "
          f"input_tokens={resp.usage.input_tokens} output_tokens={resp.usage.output_tokens}")
    for b in resp.content:
        if b.type == "text":
            print("   ", b.text[:300].replace("\n", " "))

def text_of(resp):
    return "".join(b.text for b in resp.content if b.type == "text")

print("model:", config.MODEL, "| fast model:", config.FAST_MODEL)

## Drill 1: cache a long prefix
We generate a long synthetic policy handbook. Short prompts are not cached, and the minimum length varies by model.

In [ ]:
random.seed(3)
topics = ["refunds", "outages", "device returns", "billing disputes", "plan changes", "SIM replacement", "roaming", "late fees"]
clauses = []
for i in range(1, 121):
    t = topics[i % len(topics)]
    clauses.append(f"Clause {i} ({t}): Agents must confirm the account ID, record the customer's stated issue, check the last three "
                   f"bills for related charges, and apply policy rule {t.upper()}-{i:03d} before offering any credit. Credits above "
                   f"{random.choice([25, 50, 75, 100])} USD need supervisor approval, and every exception is logged with a reason code.")
HANDBOOK = "TELECOM SUPPORT POLICY HANDBOOK (synthetic)\n" + "\n".join(clauses)
n = client.messages.count_tokens(model=config.MODEL, system=HANDBOOK, messages=[{"role": "user", "content": "hi"}]).input_tokens
print("handbook size:", n, "tokens")

In [ ]:
def ask(question, system_blocks):
    t0 = time.perf_counter()
    r = client.messages.create(model=config.MODEL, max_tokens=200, system=system_blocks,
                               messages=[{"role": "user", "content": question}])
    u = r.usage
    print(f"  {time.perf_counter() - t0:5.2f}s  input={u.input_tokens:6d}  cache_write={u.cache_creation_input_tokens or 0:6d}  "
          f"cache_read={u.cache_read_input_tokens or 0:6d}")
    return r
CACHED = [{"type": "text", "text": HANDBOOK, "cache_control": {"type": "ephemeral"}}]
print("call 1 (expect a cache write):"); ask("Which clause covers roaming credits?", CACHED)
print("call 2 (expect a cache read):");  ask("What approval is needed for a 100 USD credit?", CACHED)
print("call 3 (expect a cache read):");  ask("Summarize the SIM replacement rules.", CACHED)

`cache_read_input_tokens` above zero is the proof that caching works. Reads are billed at a fraction of normal input.

## Drill 2: break the cache
**Predict:** what happens when the current time is placed before the handbook?

In [ ]:
from datetime import datetime
for i in range(2):
    stamp = f"Current time: {datetime.now().isoformat(timespec='microseconds')}\n"
    print(f"timestamp first, call {i + 1}:")
    ask("Which clause covers late fees?", [{"type": "text", "text": stamp + HANDBOOK, "cache_control": {"type": "ephemeral"}}])

In [ ]:
# Fix: stable prefix first and cached; changing details after the breakpoint
for i in range(2):
    stamp = f"Current time: {datetime.now().isoformat(timespec='microseconds')}"
    print(f"handbook first, call {i + 1}:")
    ask("Which clause covers late fees?", CACHED + [{"type": "text", "text": stamp}])

## Drill 3: a Message Batch
Batches cost 50% of standard prices, most finish within an hour, and there is no latency guarantee. One request below is deliberately broken.

In [ ]:
REQUESTS = {f"ticket-{i:02d}": text for i, text in enumerate([
    "Router drops every evening.", "Charged twice in September.", "Need to change billing email.",
    "No signal in basement office.", "Want to cancel roaming pack.", "App shows wrong data balance."], 1)}
reqs = [{"custom_id": cid, "params": {"model": config.FAST_MODEL, "max_tokens": 120,
          "messages": [{"role": "user", "content": f"Classify this telecom ticket in 3 words: {txt}"}]}} for cid, txt in REQUESTS.items()]
reqs.append({"custom_id": "ticket-broken", "params": {"model": "claude-model-that-does-not-exist", "max_tokens": 50,
             "messages": [{"role": "user", "content": "hi"}]}})
batch = client.messages.batches.create(requests=reqs)
print("batch", batch.id, batch.processing_status)

In [ ]:
deadline, results = time.time() + 600, {}
while time.time() < deadline:
    b = client.messages.batches.retrieve(batch.id)
    print("status:", b.processing_status, b.request_counts)
    if b.processing_status == "ended":
        break
    time.sleep(20)
if b.processing_status == "ended":
    for entry in client.messages.batches.results(batch.id):   # any order: match by custom_id
        results[entry.custom_id] = entry.result
    for cid in sorted(results):
        r = results[cid]
        print(f"{cid:14s} {r.type:10s}", text_of(r.message).strip() if r.type == "succeeded" else "")
    failed = [cid for cid, r in results.items() if r.type != "succeeded"]
    print("\nresubmit only these, after fixing them:", failed)
else:
    print("Still processing. That is normal: batches have no latency guarantee. Rerun this cell later.")

## Drill 4: cost worksheet
Copy current per-million-token prices for your model from the official pricing page. Multipliers follow the caching and batch docs; confirm them there too.

In [ ]:
PRICE_IN_PER_MTOK = None     # e.g. from the pricing page
PRICE_OUT_PER_MTOK = None
CACHE_READ_X, CACHE_WRITE_X, BATCH_X = 0.1, 1.25, 0.5
REQUESTS_PER_DAY, PREFIX_TOKENS, QUESTION_TOKENS, OUTPUT_TOKENS = 5000, 20000, 200, 300
if PRICE_IN_PER_MTOK and PRICE_OUT_PER_MTOK:
    pin, pout = PRICE_IN_PER_MTOK / 1e6, PRICE_OUT_PER_MTOK / 1e6
    std = REQUESTS_PER_DAY * ((PREFIX_TOKENS + QUESTION_TOKENS) * pin + OUTPUT_TOKENS * pout)
    cached = (PREFIX_TOKENS * CACHE_WRITE_X * pin) + REQUESTS_PER_DAY * (PREFIX_TOKENS * CACHE_READ_X * pin + QUESTION_TOKENS * pin + OUTPUT_TOKENS * pout)
    print(f"standard:        ${std:,.2f}/day")
    print(f"cached prefix:   ${cached:,.2f}/day (assumes the cache stays warm)")
    print(f"batch only:      ${std * BATCH_X:,.2f}/day")
    print(f"batch + cache:   ${cached * BATCH_X:,.2f}/day (approximate; discounts combine)")
else:
    print("Fill in both prices to compare. Which lever saves the most for this workload, and why?")

## You should now have
- [ ] Usage showing a cache write, then cache reads
- [ ] A broken cache and its fix
- [ ] A batch matched by custom_id, with failures listed
- [ ] A cost comparison for one of your workloads

## Check yourself
1. Which usage field proves a cache hit?
2. Why did the timestamp break the cache?
3. Why must batch results be matched by custom_id?